In [1]:
import enum
import json
import os
from copy import deepcopy

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch.nn.functional import softmax
from torch.optim import AdamW
from torch.utils.data import DataLoader

from internal.data_types import HistologyDataset
from internal.nn.mixup_cutmix_wrapper import MixupCutmixWrapper
from internal.nn.model import train_one_epoch, validate
from internal.nn.test_time_augmentation import apply_multicrop_tta, apply_mask_multicrop_tta
from internal.nn.test_time_augmentation import apply_tta
from internal.nn.weighted_random_sampler import make_weighted_sampler
from internal.persistence_manager import PersistenceManager

data = PersistenceManager.load_dataset()
test_df = data.test_df
train_df = data.train_df
train_transforms = data.train_transforms
val_test_transforms = data.val_test_transforms
idx2label = data.idx2label

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cuda_is_available = torch.cuda.is_available()
print(f'Using device: {device}')

Arrays and scalers loaded successfully from: /home/andre/university/AN2DL-Challenge-2/notebooks/processed/dataset.joblib
Using device: cuda


In [2]:
def get_classifier_module(model: nn.Module):
    # Common names in timm models
    for name in ["classifier", "fc", "head"]:
        if hasattr(model, name):
            return getattr(model, name), name
    # Fallback: assume there is a single linear at the very end
    last_linear = None
    for m in reversed(list(model.modules())):
        if isinstance(m, nn.Linear):
            last_linear = m
            break
    if last_linear is None:
        raise RuntimeError("Could not find classifier layer in model.")
    return last_linear, None

In [3]:
class PreTrainedArchitectures(enum.Enum):
    EFFICIENTNETV2_S = "tf_efficientnetv2_s.in21k"
    EFFICIENTNET_B1_NS = "tf_efficientnet_b1.ns_jft_in1k"
    CONVNEXT_TINY = "convnext_tiny"
    EFFICIENTNET_B0 = "efficientnet_b0"
    EFFICIENTNET_B1 = "efficientnet_b1"

MODEL_TO_USE: PreTrainedArchitectures = PreTrainedArchitectures.EFFICIENTNET_B1_NS

In [4]:
best_f1_per_fold: dict[int, int] = {}
N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
N_CLASSES = 4  # number of classes in the dataset (labels)

# efficientnet_b0 / efficientnet_b1

In [5]:
def create_efficientnet_b0_model(pretrained: bool = True) -> nn.Module:
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=pretrained,
        num_classes=N_CLASSES,
        in_chans=4,
        drop_rate=0.3,          # Dropout
        drop_path_rate=0.1      # Stochastic depth
    ).to(device)
    return model

def freeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False

def unfreeze_last_two_blocks_and_head(model: nn.Module):
    """
    For EfficientNet from timm: unfreeze last 2 blocks + classifier head.
    """
    freeze_all(model)

    # Last 2 conv blocks
    if hasattr(model, "blocks"):
        for blk in model.blocks[-2:]:
            for p in blk.parameters():
                p.requires_grad = True

    # Classifier head
    clf_module, _ = get_classifier_module(model)
    for p in clf_module.parameters():
        p.requires_grad = True


if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B0 or MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B1:
    N_FOLDS = data.num_K_folds
    IMAGE_SIZE = data.image_size
    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 2
    PRETRAINED_MODEL = MODEL_TO_USE.value
    N_CLASSES = 4 # number of classes in the dataset (labels)
    N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
    EPOCHS = 25
    LR = 3e-4
    PREFIX = "effb0" if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B0 else "effb1"

    for fold in range(N_FOLDS):
        print(f"\n========== Fold {fold} ==========")

        train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

        train_dataset = HistologyDataset(
            df=train_df_split,
            image_size=IMAGE_SIZE,
            is_train=True,  # Augmentations applied
            use_mask_crop=True
        )
        val_dataset = HistologyDataset(
            df=val_df_split,
            image_size=IMAGE_SIZE,
            is_train=False, # Disable augmentations
            use_mask_crop=True
        )

        sampler = make_weighted_sampler(train_df_split)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        val_loader   = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        # ---- create model + unfreeze last 2 blocks + head ----
        model = create_efficientnet_b0_model(pretrained=True)
        unfreeze_last_two_blocks_and_head(model)

        # ---- loss, optimizer, scheduler ----
        class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32)
        class_weights = (class_counts.sum() / class_counts)
        class_weights = class_weights / class_weights.mean()

        criterion = nn.CrossEntropyLoss(
            weight=class_weights.to(device),
            label_smoothing=0.1
        )

        optimizer = AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR,
            weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS
        )

        # ---- training loop ----
        best_f1 = 0.0
        best_state = None
        mixup_fn = MixupCutmixWrapper(
            alpha=0.4,       # mixup/cutmix Beta distribution
            mixup_prob=0.4,  # 40% of batches => mixup
            cutmix_prob=0.2  # 20% of batches => cutmix
        )

        for epoch in range(1, EPOCHS + 1):
            print(f"\nEpoch {epoch}/{EPOCHS}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device, grad_accum_steps=GRAD_ACCUM_STEPS, mixup_fn=mixup_fn
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()

            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )

            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = deepcopy(model.state_dict())
                torch.save(
                    best_state,
                    f"best_effb0_fold{fold}_f1_{val_f1:.4f}.pth"
                )
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        # restore best weights for this fold
        if best_state is not None:
            model.load_state_dict(best_state)
            print(f"Restored best weights for fold {fold} (F1={best_f1:.4f})")

        # save final model for inference
        torch.save(model.state_dict(), f"effb0_fold{fold}.pth")

        # record best F1 for this fold
        best_f1_per_fold[fold] = best_f1

# tf_efficientnet_b1_ns

In [6]:
def create_efficientnet_b1_ns_model(pretrained: bool = True) -> nn.Module:
    model = timm.create_model(
        MODEL_TO_USE.value,
        pretrained=pretrained,
        num_classes=N_CLASSES,
        in_chans=4,          # 3 RGB + 1 mask
        drop_rate=0.3,       # stronger dropout than B0
        drop_path_rate=0.1   # stochastic depth
    ).to(device)
    return model

def freeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False

def unfreeze_last_two_blocks_and_head(model: nn.Mefficientnet_b0odule):
    """
    Freeze earlier EfficientNet blocks, unfreeze the last two + head.
    Works for timm tf_efficientnet_b* models.
    """
    # 1) Freeze everything by default
    for p in model.parameters():
        p.requires_grad = False

    # 2) Unfreeze last two blocks
    # model.blocks is a nn.Sequential
    num_blocks = len(model.blocks)
    for idx in range(num_blocks - 2, num_blocks):
        for p in model.blocks[idx].parameters():
            p.requires_grad = True

    # 3) Unfreeze conv_head + bn2 + classifier
    for p in model.conv_head.parameters():
        p.requires_grad = True
    for p in model.bn2.parameters():
        p.requires_grad = True
    for p in model.classifier.parameters():
        p.requires_grad = True

def unfreeze_last_stage_and_head(model: nn.Module):
    """
    EfficientNet B1-NS recommended fine-tuning strategy:
    - Freeze all early MBConv stages
    - Unfreeze the last MBConv stage (stage 6)
    - Unfreeze conv_head + bn2 + classifier
    """

    # Freeze everything first
    for p in model.parameters():
        p.requires_grad = False

    # ---- Unfreeze last stage (stage 6) ----
    # EfficientNet blocks are sequential but grouped in stages.
    # B1 layout roughly:
    #   Stage0: stem
    #   Stage1: blocks[0]
    #   Stage2: blocks[1:3]
    #   Stage3: blocks[3:5]
    #   Stage4: blocks[5:8]
    #   Stage5: blocks[8:11]
    #   Stage6: blocks[11:15]  <-- last stage
    last_stage_start = len(model.blocks) - 4  # 4 blocks in last stage (B1)
    for idx in range(last_stage_start, len(model.blocks)):
        for p in model.blocks[idx].parameters():
            p.requires_grad = True

    # ---- Unfreeze head ----
    for p in model.conv_head.parameters():
        p.requires_grad = True
    for p in model.bn2.parameters():
        p.requires_grad = True
    for p in model.classifier.parameters():
        p.requires_grad = True

if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B1_NS:
    N_FOLDS = data.num_K_folds
    IMAGE_SIZE = data.image_size
    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 2
    PRETRAINED_MODEL = MODEL_TO_USE.value
    N_CLASSES = 4 # number of classes in the dataset (labels)
    N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
    EPOCHS = 25
    LR = 7.5e-4
    WEIGHT_DECAY = 2e-4
    PREFIX = "tf_effb1_ns"

    for fold in range(N_FOLDS):
        print(f"\n========== Fold {fold} ==========")

        train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

        train_dataset = HistologyDataset(
            df=train_df_split,
            image_size=IMAGE_SIZE,
            is_train=True,  # Augmentations applied
            use_mask_crop=True
        )
        val_dataset = HistologyDataset(
            df=val_df_split,
            image_size=IMAGE_SIZE,
            is_train=False, # Disable augmentations
            use_mask_crop=True
        )

        sampler = make_weighted_sampler(train_df_split)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        val_loader   = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        # ---- create model + unfreeze last 2 blocks + head ----
        model = create_efficientnet_b1_ns_model(pretrained=True)
        unfreeze_last_stage_and_head(model)

        # ---- loss, optimizer, scheduler ----
        class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32)
        class_weights = (class_counts.sum() / class_counts)
        class_weights = class_weights / class_weights.mean()

        criterion = nn.CrossEntropyLoss(
            weight=class_weights.to(device),
            label_smoothing=0.05
        )

        optimizer = AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR,
            weight_decay=WEIGHT_DECAY
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS
        )

        # ---- training loop ----
        best_f1 = 0.0
        best_state = None
        mixup_fn = MixupCutmixWrapper(
            alpha=0.3,       # mixup/cutmix Beta distribution
            mixup_prob=0.3,  # 30% of batches => mixup
            cutmix_prob=0.2  # 20% of batches => cutmix
        )

        for epoch in range(1, EPOCHS + 1):
            print(f"\nEpoch {epoch}/{EPOCHS}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device, grad_accum_steps=GRAD_ACCUM_STEPS, mixup_fn=mixup_fn
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()

            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )

            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = deepcopy(model.state_dict())
                torch.save(
                    best_state,
                    f"best_{PREFIX}_fold{fold}_f1_{val_f1:.4f}.pth"
                )
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        # restore best weights for this fold
        if best_state is not None:
            model.load_state_dict(best_state)
            print(f"Restored best weights for fold {fold} (F1={best_f1:.4f})")

        # save final model for inference
        torch.save(model.state_dict(), f"{PREFIX}_fold{fold}.pth")

        # record best F1 for this fold
        best_f1_per_fold[fold] = best_f1


========== Fold 0 ==========

Epoch 1/25


    t_loss=2.1702 | F1(macro)=0.2939 | Acc=0.2996


Confusion matrix:
 [[ 5  0  1 35]
 [ 6  0  2 24]
 [ 3  1  2 24]
 [ 0  0  1 13]]
Train  loss=2.1702 acc=0.2996 f1=0.2939 | Val loss=2.5653 acc=0.1709 f1=0.1323
  🔥 New best F1: 0.1323 – model saved.

Epoch 2/25


    t_loss=1.6350 | F1(macro)=0.3309 | Acc=0.3384


Confusion matrix:
 [[ 6  3 10 22]
 [ 3  4  7 18]
 [ 3  6 10 11]
 [ 1  1  1 11]]
Train  loss=1.6350 acc=0.3384 f1=0.3309 | Val loss=1.8752 acc=0.2650 f1=0.2576
  🔥 New best F1: 0.2576 – model saved.

Epoch 3/25


...

Confusion matrix:
 [[21  6  7  6]
 [11 12  6  3]
 [11  4 11  4]
 [ 3  4  6  1]]
Train  loss=0.6830 acc=0.7871 f1=0.7826 | Val loss=1.9272 acc=0.3879 f1=0.3351

Epoch 24/25


    t_loss=0.6512 | F1(macro)=0.7740 | Acc=0.7828


Confusion matrix:
 [[17  8 11  4]
 [ 9 11  9  3]
 [ 6  8 15  1]
 [ 4  3  6  1]]
Train  loss=0.6512 acc=0.7828 f1=0.7740 | Val loss=2.0523 acc=0.3793 f1=0.3279

Epoch 25/25


    t_loss=0.6507 | F1(macro)=0.7788 | Acc=0.7849


Confusion matrix:
 [[19  9  7  5]
 [ 9 13  8  2]
 [ 7  7 12  4]
 [ 5  3  5  1]]
Train  loss=0.6507 acc=0.7849 f1=0.7788 | Val loss=1.8986 acc=0.3879 f1=0.3363
Restored best weights for fold 4 (F1=0.3578)


# tf_efficientnetv2_s.in21k

In [7]:
def create_model_tf_efficientnetv2_s(pretrained: bool = True) -> nn.Module:
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=pretrained,
        num_classes=N_CLASSES,
        in_chans=4,
        drop_rate=0.3,        # Dropout
        drop_path_rate=0.1    # Stochastic depth
    ).to(device)
    return model

if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S:
    N_FOLDS = data.num_K_folds
    IMAGE_SIZE = data.image_size
    BATCH_SIZE = 4
    PRETRAINED_MODEL = MODEL_TO_USE.value
    N_CLASSES = 4 # number of classes in the dataset (labels)
    N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
    EPOCHS_STAGE1 = 10
    EPOCHS_STAGE2 = 15

    for fold in range(N_FOLDS):
        print(f"\n========== Fold {fold} ==========")

        train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

        train_dataset = HistologyDataset(
            df=train_df_split,
            image_size=IMAGE_SIZE,
            is_train=True,
            use_mask_crop=True
        )
        val_dataset = HistologyDataset(
            df=val_df_split,
            image_size=IMAGE_SIZE,
            is_train=False,   # False to disable augmentations
            use_mask_crop=True
        )

        sampler = make_weighted_sampler(train_df_split)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        val_loader   = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )

        # --- create fresh model for this fold ---
        model = create_model_tf_efficientnetv2_s()

        # --- Stage 1: freeze backbone, train classifier head ---
        print("\n--- Stage 1: Training classifier head ---")

        # --- 1.1. freeze feature extractor layers ---
        for param in model.parameters():
            param.requires_grad = False

        # 2) unfreeze classifier head (EffNetV2 uses .classifier)
        for param in model.classifier.parameters():
            param.requires_grad = True

        # --- 1.2. define loss, optimizer, scheduler ---
        criterion = nn.CrossEntropyLoss()
        head_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(head_params, lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE1)

        # --- 1.3. train for several epochs ---
        best_f1 = 0.0
        best_state = None
        for epoch in range(1, EPOCHS_STAGE1+1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE1}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, f"best_effv2_stage1_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)
            print(f"Restored best Stage 1 weights for fold {fold} (F1={best_f1:.4f})")

        # --- Stage 2: unfreeze whole model, fine-tune ---
        print("\n--- Stage 2: Fine-tuning entire model ---")

        # --- 2.1. unfreeze entire model ---
        for param in model.parameters():
            param.requires_grad = True

        # --- 2.2. define loss, optimizer, scheduler ---
        optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE2)

        # --- 2.3. mild class weights ---
        class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32) # 445 LumB, 414 LumA, 397 Her2, 156 TN
        class_weights = (class_counts.sum() / class_counts)
        class_weights = class_weights / class_weights.mean()
        # criterion = FocalLoss(alpha=class_weights, gamma=2.0)
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)

        # --- 2.4. train for several epochs ---
        best_f1 = 0.0
        best_state = None

        for epoch in range(1, EPOCHS_STAGE2 + 1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE2}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_f1_per_fold[fold] = best_f1
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, f"best_effv2_stage2_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)   # restore best val-F1 weights

        # --- save model for this fold ---
        torch.save(model.state_dict(), f"effv2_s_fold{fold}.pth")

# convnext_tiny

In [8]:
def create_model_convnext(pretrained: bool = True) -> nn.Module:
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=pretrained,
        num_classes=N_CLASSES,
        in_chans=4,
        drop_rate=0.3,        # Dropout
        drop_path_rate=0.1    # Stochastic depth
    ).to(device)
    return model

if MODEL_TO_USE == PreTrainedArchitectures.CONVNEXT_TINY:
    N_FOLDS = data.num_K_folds
    IMAGE_SIZE = data.image_size
    BATCH_SIZE = 4
    PRETRAINED_MODEL = MODEL_TO_USE.value
    N_CLASSES = 4  # number of classes in the dataset (labels)
    N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
    EPOCHS_STAGE1 = 8
    EPOCHS_STAGE2 = 12

    for fold in range(N_FOLDS):
        print(f"\n========== Fold {fold} ==========")

        train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

        train_dataset = HistologyDataset(
            df=train_df_split,
            image_size=IMAGE_SIZE,
            is_train=True,
            use_mask_crop=True
        )
        val_dataset = HistologyDataset(
            df=val_df_split,
            image_size=IMAGE_SIZE,
            is_train=False,   # Disable augmentations
            use_mask_crop=True
        )

        sampler = make_weighted_sampler(train_df_split)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )

        # --- create fresh model for this fold ---
        model = create_model_convnext()

        # --- Stage 1: freeze backbone, train classifier HEAD (ConvNeXt) ---
        print("\n--- Stage 1: Training classifier head (ConvNeXt-Tiny) ---")

        # --- 1.1. freeze feature extractor layers ---
        for p in model.parameters():
            p.requires_grad = False

        # --- 1.1. unfreeze only the classifier head (ConvNeXt uses .head) ---
        for p in model.head.parameters():
            p.requires_grad = True

        # --- 1.2. define loss, optimizer, scheduler ---
        criterion = nn.CrossEntropyLoss()
        head_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(head_params, lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE1)

        # --- 1.3. train for several epochs ---
        best_f1 = 0.0
        best_state = None
        for epoch in range(1, EPOCHS_STAGE1 + 1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE1}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, f"best_convnext_stage1_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)
            print(f"Restored best Stage 1 weights for fold {fold} (F1={best_f1:.4f})")

        # --- Stage 2: unfreeze whole model, fine-tune ---
        print("\n--- Stage 2: Fine-tuning entire model (ConvNeXt-Tiny) ---")

        # --- 2.1. unfreeze entire model ---
        for p in model.parameters():
            p.requires_grad = True

        # --- 2.2. define loss, optimizer, scheduler ---
        optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE2)

        # --- 2.3. mild class weights ---
        class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32)  # 445 LumB, 414 LumA, 397 Her2, 156 TN
        class_weights = (class_counts.sum() / class_counts)
        class_weights = class_weights / class_weights.mean()
        # criterion = FocalLoss(alpha=class_weights, gamma=2.0)
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)

        # --- 2.4. train for several epochs ---
        best_f1 = 0.0
        best_state = None
        for epoch in range(1, EPOCHS_STAGE2 + 1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE2}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_f1_per_fold[fold] = best_f1
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, f"best_convnext_stage2_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)  # restore best val-F1 weights

        # --- save model for this fold ---
        torch.save(model.state_dict(), f"convnext_tiny_fold{fold}.pth")

# Model Inference with 5-Fold Ensembling

In [9]:
prefix_filename = "effv2_s" if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S else "convnext_tiny" if MODEL_TO_USE == PreTrainedArchitectures.CONVNEXT_TINY else "effb0" if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B0 else "effb1" if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B1 else "tf_effb1_ns"

FOLD_VAL_F1 = f"fold_val_f1_{prefix_filename}.json"

# Save best F1 per fold to JSON
with open(FOLD_VAL_F1, "w") as f:
    json.dump(best_f1_per_fold, f, indent=2)

In [10]:
########################################################
# ===== Inference with TTA and 5-Fold Ensembling ===== #
########################################################
all_fold_probs = []
all_sample_indices = None

test_dataset = HistologyDataset(
    df=test_df,
    image_size=data.image_size,
    is_train=False,   # deterministic, returns (img, sample_index)
    use_mask_crop=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=1,               # per-image TTA
    shuffle=False,
    num_workers=N_WORKERS,
    pin_memory=cuda_is_available
)

if os.path.exists(FOLD_VAL_F1):
    with open(FOLD_VAL_F1, "r") as f:
        best_f1_per_fold = json.load(f)
    val_f1_per_fold = np.array([best_f1_per_fold[str(k)] for k in range(data.num_K_folds)])
    # Normalize to get weights that sum to 1
    fold_weights = val_f1_per_fold / val_f1_per_fold.sum()
else:
    # fallback: uniform weights if metrics are missing
    print('Warning: fold validation F1 scores not found, using uniform weights.')
    fold_weights = np.ones(data.num_K_folds, dtype=np.float32) / data.num_K_folds

print("Fold weights:", fold_weights)

# -----------------------------
# 2) Accumulate weighted probs
# -----------------------------
all_probs = None
all_sample_indices = None

for fold in range(data.num_K_folds):
    print(f"Inference with fold {fold} model (weight={fold_weights[fold]:.3f})")

    if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S:
        model = create_model_tf_efficientnetv2_s(pretrained=False)
    elif MODEL_TO_USE == PreTrainedArchitectures.CONVNEXT_TINY:
        model = create_model_convnext(pretrained=False)
    elif MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B1 or MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B0:
        model = create_efficientnet_b0_model(pretrained=False)
    else:
        model = create_efficientnet_b1_ns_model(pretrained=False)

    state_dict = torch.load(f"{prefix_filename}_fold{fold}.pth", map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    fold_probs = []
    sample_indices_list = []

    with torch.no_grad():
        for img_tensor, sample_idx in test_loader:
            # img_tensor: [1, 4, H, W]  (RGB+mask)
            img_tensor = img_tensor.squeeze(0).to(device)  # [4, H, W]

            # -------- TTA: mask-based multi-crop + simple flips --------
            USE_MASK_TTA = False
            if USE_MASK_TTA:
                tta_tensors = apply_mask_multicrop_tta(img_tensor, crop_size=data.image_size, n_crops=2)
            else:
                tta_tensors = apply_multicrop_tta(img_tensor, base_size=data.image_size, inner_ratio=0.8)


            # accumulate probability predictions
            probs_sum = 0.0
            for aug_img in tta_tensors:
                aug_img = aug_img.unsqueeze(0).to(device)  # [1, 4, H, W]
                with torch.no_grad():
                    logits = model(aug_img)
                    probs = softmax(logits, dim=1)  # [1, N_CLASSES]
                probs_sum += probs[0].cpu().numpy()

            # average across TTA views
            avg_probs = probs_sum / len(tta_tensors)
            fold_probs.append(avg_probs)

            # collect sample indices only once
            if all_sample_indices is None:
                sample_indices_list.append(sample_idx[0])

    fold_probs = np.vstack(fold_probs)  # [N_test, N_CLASSES]

    # initialize global probs
    if all_probs is None:
        all_probs = np.zeros_like(fold_probs, dtype=np.float32)

     # weighted accumulation
    all_probs += fold_weights[fold] * fold_probs

    if all_sample_indices is None:
        all_sample_indices = sample_indices_list

# -----------------------------
# 3) Final predictions
# -----------------------------
pred_indices = all_probs.argmax(axis=1)
pred_labels = [idx2label[int(i)] for i in pred_indices]

sample_index_with_ext = [
    f"{si}.png" if not si.endswith(".png") else si
    for si in all_sample_indices
]

submission_df = pd.DataFrame({
    "sample_index": sample_index_with_ext,
    "label": pred_labels
})
submission_df.to_csv(f"submission_5fold_tta_{prefix_filename}.csv", index=False)

print(f"Saved submission_5fold_tta_{prefix_filename}.csv")

Fold weights: [0.18862717 0.20449834 0.19493961 0.20695047 0.20498441]
Inference with fold 0 model (weight=0.189)
Inference with fold 1 model (weight=0.204)
Inference with fold 2 model (weight=0.195)
Inference with fold 3 model (weight=0.207)
Inference with fold 4 model (weight=0.205)
Saved submission_5fold_tta_tf_effb1_ns.csv


In [11]:
def predict_loader_with_tta(model, loader, device):
    model.eval()
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for imgs, labels in loader:  # note: here we have labels, not sample_index
            imgs = imgs.squeeze(0).to(device)  # if batch_size=1
            tta_imgs = apply_tta(imgs)         # same apply_tta as for test

            probs_sum = 0
            for aug in tta_imgs:
                aug = aug.unsqueeze(0).to(device)
                logits = model(aug)
                probs = softmax(logits, dim=1)
                probs_sum += probs[0].cpu().numpy()

            avg_probs = probs_sum / len(tta_imgs)
            all_probs.append(avg_probs)
            all_targets.append(labels.item())

    all_probs = np.vstack(all_probs)
    all_targets = np.array(all_targets)
    pred_indices = all_probs.argmax(axis=1)

    macro_f1 = f1_score(all_targets, pred_indices, average="macro")
    return macro_f1

fold_f1s = []

for fold in range(data.num_K_folds):
    print(f"OOF eval for fold {fold}")

    # build val_df_split for that fold
    val_df_split = train_df[train_df["fold"] == fold].reset_index(drop=True)
    val_dataset = HistologyDataset(
        df=val_df_split,
        image_size=data.image_size,
        is_train=False,   # Disable augmentations
        use_mask_crop=True
    )
    val_loader  = DataLoader(val_dataset, batch_size=1, shuffle=False,
                             num_workers=N_WORKERS, pin_memory=cuda_is_available)

    if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S:
        model = create_model_tf_efficientnetv2_s(pretrained=False)
    elif MODEL_TO_USE == PreTrainedArchitectures.CONVNEXT_TINY:
        model = create_model_convnext(pretrained=False)
    else:
        model = create_efficientnet_b0_model(pretrained=False)
    model.load_state_dict(torch.load(f"{prefix_filename}_fold{fold}.pth", map_location=device))

    f1 = predict_loader_with_tta(model, val_loader, device)
    fold_f1s.append(f1)
    print("Fold F1 (OOF, with TTA):", f1)

print("Mean OOF F1:", np.mean(fold_f1s))


OOF eval for fold 0
Fold F1 (OOF, with TTA): 0.3316949152542373
OOF eval for fold 1
Fold F1 (OOF, with TTA): 0.2756870586233302
OOF eval for fold 2
Fold F1 (OOF, with TTA): 0.277041302172585
OOF eval for fold 3
Fold F1 (OOF, with TTA): 0.365990778420157
OOF eval for fold 4
Fold F1 (OOF, with TTA): 0.3307377365281777
Mean OOF F1: 0.3162303581996974
